# 08 · Everything Is an Object: Objects, Classes, `type` & Metaclasses

**Module 06 – Object-Oriented Programming** · Estimated time: 90 min

You have probably heard "**in Python, everything is an object**". This lesson takes that sentence apart slowly, from zero, until nothing about it feels like magic. We will answer, one by one:

1. What *is* an object, really?
2. Is it true that *everything* is an object? (And what is **not**?)
3. If every value is an object, **who creates** it, and **how**?
4. Objects are made by classes… so what makes a **class**?
5. What is `type`, what is `object`, and why does `type(type)` say `type`?
6. What is a **metaclass**, and how do custom classes and metaclasses fit together?

### 🎯 What you will learn
- The three things every object has: **identity**, **type** and **value**
- The difference between a **name** (variable) and an **object**, and why `is` ≠ `==`
- Proof that numbers, strings, `None`, functions, modules, classes and even `type` are objects
- How an object is born step by step: `Class(...)` → `__call__` → `__new__` → `__init__`
- Why classes are objects too, and that `class` statements secretly call `type(name, bases, namespace)`
- The `object` ↔ `type` relationship: "instance of" versus "subclass of"
- What a metaclass is, how to write one, and the full class-creation sequence
- The simpler tools (`__init_subclass__`, class decorators) you should usually use instead

> 💡 **Prerequisites:** Module 06 · 01 (classes and `__init__`) and 06 · 02 (inheritance). Deeper follow-ups: Module 10 · 01 (data model and attribute lookup) and 10 · 03 (metaclasses in depth).

## 1. What is an object?

An **object** is a chunk of data living in your computer's memory that Python manages for you. Every single object, no matter how simple, carries exactly three things:

| Part | Question it answers | How to see it | Can it change? |
|---|---|---|---|
| **Identity** | *Which* object is this? (its unique ID) | `id(x)`, compared with `is` | Never, for the object's whole life |
| **Type** | *What kind* of object is it? What can it do? | `type(x)` | Practically never |
| **Value** | *What data* does it hold? | printing it, compared with `==` | Only if the type is **mutable** |

Think of an object as a **box in a warehouse**:
- the **identity** is the box's serial number stamped on it,
- the **type** is the label saying what kind of box it is ("int box", "list box") and therefore what you are allowed to do with it,
- the **value** is what is inside.

In [1]:
x = 42
print("identity:", id(x))      # a unique number (in CPython it is the memory address)
print("type:    ", type(x))
print("value:   ", x)

identity: 4357290016
type:     <class 'int'>
value:    42


The exact `id` number will be different on your machine and every time you run it. That is fine: what matters is that **two different objects alive at the same time never share an id**.

> 💡 In CPython (the normal Python from python.org), `id()` happens to be the memory address. That is an implementation detail; treat it only as "a unique number".

## 2. Names are not objects: they are labels

This is the single most important idea in the lesson. A **variable is not a box**. A variable is a **name tag** tied to a box. The box (object) lives on its own; names just point to it.

```
   names (labels)            objects (boxes in memory)
   ──────────────            ─────────────────────────
     a ─────────┐
                ├──────────▶  [1, 2, 3]   (a list object)
     b ─────────┘
```

`b = a` does **not** copy the list. It ties a second name tag onto the **same** box:

In [2]:
a = [1, 2, 3]
b = a                  # no copy! b is another label on the same object
b.append(4)
print(a)               # a "changed" too, because it is the same object
print(a is b, id(a) == id(b))

[1, 2, 3, 4]
True True


### `is` vs `==`

- `a == b` asks: **do these two boxes contain equal values?** (calls `__eq__`)
- `a is b` asks: **is this the very same box?** (compares identity)

In [3]:
c = [1, 2, 3, 4]       # a NEW list that happens to hold the same values
print(a == c)          # same value
print(a is c)          # different objects

True
False


> ❌ **Common mistake:** using `is` to compare numbers or strings (`if x is 1000:`). Whether two equal numbers are the same object is an internal optimisation that can change between versions. Python even warns you: `SyntaxWarning: "is" with 'int' literal`. Use `==` for values, and keep `is` for `None`, `True`/`False` and "is this literally the same object?" checks.

In [4]:
result = None
print(result is None)  # ✅ the correct way to test for None

True


### Mutable vs immutable objects

The **value** of some objects can change in place (**mutable**: `list`, `dict`, `set`, most of your own classes). Others can never change (**immutable**: `int`, `float`, `str`, `tuple`, `frozenset`, `bool`, `None`).

When you "change" an immutable object, Python actually builds a **new** object and moves the name tag to it:

In [5]:
n = 10
before = id(n)
n = n + 1              # creates a NEW int object 11; the name n now points there
print(n, before == id(n))   # False → different object

nums = [1, 2]
before = id(nums)
nums.append(3)         # changes the SAME list object in place
print(nums, before == id(nums))   # True → same object

11 False
[1, 2, 3] True


## 3. "Everything is an object": the proof

An object is anything that has an identity, a type and a value, and that you can put in a variable, pass to a function, or return from one. Let's test that on things that do **not** look like objects at first.

`object` is the root of every class in Python, so `isinstance(thing, object)` is our test.

In [6]:
import math

def greet():
    return "hi"

class Dog:
    pass

things = [42, 3.14, "hello", True, None, [1, 2], (1, 2), {"a": 1},
          greet, len, print, math, Dog, Dog(), int, ValueError("oops"), type, object]

for t in things:
    print(f"{type(t).__name__:28} isinstance(…, object) = {isinstance(t, object)}")

int                          isinstance(…, object) = True
float                        isinstance(…, object) = True
str                          isinstance(…, object) = True
bool                         isinstance(…, object) = True
NoneType                     isinstance(…, object) = True
list                         isinstance(…, object) = True
tuple                        isinstance(…, object) = True
dict                         isinstance(…, object) = True
function                     isinstance(…, object) = True
builtin_function_or_method   isinstance(…, object) = True
builtin_function_or_method   isinstance(…, object) = True
module                       isinstance(…, object) = True
type                         isinstance(…, object) = True
Dog                          isinstance(…, object) = True
type                         isinstance(…, object) = True
ValueError                   isinstance(…, object) = True
type                         isinstance(…, object) = True
type          

Every line says `True`. Look at what is in that list:
- **numbers, strings, `True`, `None`**: objects (`None` is the one and only object of type `NoneType`)
- **functions** (`greet`, `len`, `print`): objects of type `function` / `builtin_function_or_method`
- **a module** (`math`): an object of type `module`
- **a class** (`Dog`, `int`): an object of type `type` (we'll come back to this, it is the key to metaclasses)
- **an exception** (`ValueError("oops")`): an object
- **`type` and `object` themselves**: objects too

Because they are all objects, they all have **attributes** you can look at:

In [7]:
print(greet.__name__)          # functions know their own name
print(math.__name__, math.pi)  # modules have attributes
print(Dog.__name__)            # classes have attributes
print((42).bit_length())       # even an int literal has methods
print("hello".upper())         # and so does a str literal

greet
math 3.141592653589793
Dog
6
HELLO


### Functions really are just objects

Since a function is an object, you can do everything with it that you do with any other object: give it another name, store it in a list or dict, pass it to another function, even add your own attributes to it.

In [8]:
say = greet                      # another name tag on the same function object
print(say(), say is greet)

actions = {"greet": greet, "length": len}
print(actions["length"]("abc"))  # look a function up in a dict, then call it

greet.author = "Asha"            # attach a custom attribute to a function object
print(greet.author)
print(greet.__code__.co_name)    # even the compiled code inside is an object

hi True
3
Asha
greet


### Even operators turn into method calls on objects

`1 + 2` looks like special syntax, but Python translates it into a method call on the object on the left:

In [9]:
print(1 + 2)
print((1).__add__(2))          # what `1 + 2` really does
print(int.__add__(1, 2))       # same thing, called through the class
print([1, 2].__len__(), len([1, 2]))

3
3
3
2 2


This is why your own classes can support `+`, `len()`, `==` and friends (Module 06 · 05): you just define the matching **dunder methods**.

### So what is *not* an object?

"Everything" means **every value you can work with at runtime**. A few parts of the *language* are not values, so they are not objects:

| Not an object | Why |
|---|---|
| **Keywords and statements**: `if`, `for`, `def`, `class`, `return`, `import` | They are grammar, not data. (But what they *produce* is: `def` makes a function object, `class` makes a class object, `import` gives a module object.) |
| **Names / variables themselves** | A name is a label in a namespace; the thing it points to is the object. (The namespace itself, e.g. `globals()`, *is* a dict object.) |
| **Operators** like `+`, `==` | Syntax that Python translates into method calls on objects |

You cannot write `x = if`, but you can write `x = print`, `x = int`, `x = math`. That is the practical test.

## 4. Every object is made by a class

Here is the next big rule: **every object has exactly one type, and its type is a class**. The class is the "factory" that made it. `type(x)` (or `x.__class__`) tells you which factory.

In [10]:
for value in [5, "hi", [1], {1: 2}, None, greet]:
    print(f"{value!r:<40} was made by {type(value)}   same as __class__? {type(value) is value.__class__}")

5                                        was made by <class 'int'>   same as __class__? True
'hi'                                     was made by <class 'str'>   same as __class__? True
[1]                                      was made by <class 'list'>   same as __class__? True
{1: 2}                                   was made by <class 'dict'>   same as __class__? True
None                                     was made by <class 'NoneType'>   same as __class__? True
<function greet at 0x10a511a60>          was made by <class 'function'>   same as __class__? True


### Literals are shortcuts for calling a class

When you write `5` or `"hi"` or `[]`, Python creates the object for you. It is (in effect) the same as calling the class:

| You write | Equivalent class call | Type |
|---|---|---|
| `5` | `int("5")` | `int` |
| `3.5` | `float("3.5")` | `float` |
| `"hi"` | `str("hi")` | `str` |
| `[1, 2]` | `list((1, 2))` | `list` |
| `{"a": 1}` | `dict(a=1)` | `dict` |
| `def f(): ...` | (the interpreter builds a `function` object) | `function` |
| `class Dog: ...` | `type("Dog", (), {...})` (section 7) | `type` |

In [11]:
print(int("5") == 5, str("hi") == "hi", list((1, 2)) == [1, 2], dict(a=1) == {"a": 1})

True True True True


The literal form is just faster and nicer to read. Behind both, a class produced the object.

## 5. How an object is born: `__new__` and `__init__`

What actually happens when you write `Dog("Rex")`? Calling a class is a two-step process:

```
Dog("Rex")
   │
   ▼
type(Dog).__call__(Dog, "Rex")        ← calling a class is calling its *metaclass* (section 9)
   │
   ├─▶ 1. obj = Dog.__new__(Dog, "Rex")   CREATE: allocate an empty object
   │
   ├─▶ 2. Dog.__init__(obj, "Rex")        INITIALISE: fill in its attributes
   │
   └─▶ 3. return obj
```

- **`__new__`** is the **constructor**: it makes a brand-new, empty object and returns it. You almost never write one; `object.__new__` does the job.
- **`__init__`** is the **initialiser**: it receives that new object as `self` and sets it up. This is the one you write all the time.

Let's watch it happen:

In [12]:
class Dog:
    def __new__(cls, *args, **kwargs):
        print(f"1. __new__ : creating an empty {cls.__name__} object")
        obj = super().__new__(cls)          # object.__new__ allocates the memory
        print(f"   got a fresh, empty object: {vars(obj)}")
        return obj

    def __init__(self, name):
        print(f"2. __init__: filling in name={name!r}")
        self.name = name

rex = Dog("Rex")
print("3. done:", rex.name)

1. __new__ : creating an empty Dog object
   got a fresh, empty object: {}
2. __init__: filling in name='Rex'
3. done: Rex


You can even do the two steps **by hand**, which shows there is nothing magic about `Dog(...)`:

In [13]:
class Cat:
    def __init__(self, name):
        self.name = name

empty = Cat.__new__(Cat)          # step 1: a Cat with no attributes yet
print(vars(empty))                # {} → nothing inside
Cat.__init__(empty, "Tom")        # step 2: initialise it
print(vars(empty))

{}
{'name': 'Tom'}


> 💡 **When would you write `__new__`?** Rarely: when subclassing an **immutable** type (`int`, `str`, `tuple`), whose value must be set before `__init__` runs, or to return a cached / shared object instead of a new one. Module 10 · 03 shows both.

### The simplest object of all: `object()`

`object` is the root class. Calling it gives you the most basic object possible: it has an identity, a type, and almost nothing else. It doesn't even have room for attributes:

In [14]:
plain = object()
print(type(plain), isinstance(plain, object))
try:
    plain.color = "red"
except AttributeError as e:
    print("AttributeError:", e)

<class 'object'> True
AttributeError: 'object' object has no attribute 'color' and no __dict__ for setting new attributes


Your own classes *do* accept new attributes because each instance gets a `__dict__`, a dictionary that stores its attributes.

## 6. Where an object keeps its data: `__dict__`

Every normal object stores its own attributes in a dict called `__dict__`. Its **class** is also an object with its own `__dict__`, which holds the methods and class attributes shared by all instances.

In [15]:
class Car:
    wheels = 4                          # class attribute: lives on the class

    def __init__(self, brand):
        self.brand = brand              # instance attribute: lives on the object

    def honk(self):
        return f"{self.brand}: beep!"

car = Car("Tata")
print("instance dict:", car.__dict__)
print("class dict   :", [k for k in Car.__dict__ if not k.startswith("__")])

instance dict: {'brand': 'Tata'}
class dict   : ['wheels', 'honk']


When you write `car.wheels`, Python searches in order:

```
car.wheels
  1. car.__dict__          → not here
  2. Car.__dict__          → found: 4
  3. parent classes (MRO)  → (only if still not found)
```

So the method `honk` is stored **once** on the class, and `car.honk()` finds it there and passes `car` in as `self`:

In [16]:
print(car.wheels, car.honk())
print(Car.honk(car))                 # what car.honk() really does

4 Tata: beep!
Tata: beep!


The full lookup rules (including properties and descriptors) are in Module 10 · 01.

## 7. Classes are objects too

Now the twist. If *everything* is an object, then a **class** must also be an object. It is! A `class` statement creates a **class object** and binds a name to it, just like `x = 5` binds a name to an int object.

So you can do with a class everything you can do with any object:

In [17]:
class Robot:
    def __init__(self, name):
        self.name = name

print(id(Robot), type(Robot))          # a class has an identity and a type

Machine = Robot                        # another name for the same class object
r = Machine("R2")
print(type(r).__name__, isinstance(r, Robot))

def build(factory, name):              # pass a class into a function
    return factory(name)
print(build(Robot, "C3PO").name)

kinds = [int, str, Robot]              # store classes in a list
print([k.__name__ for k in kinds])

Robot.version = 2                      # add an attribute to the class object at runtime
print(r.version)                       # instances see it through the class

35971128336 <class 'type'>
Robot True
C3PO
['int', 'str', 'Robot']
2


### The `class` statement runs like normal code

A class body is not a declaration that Python reads ahead of time. It is **code that runs, top to bottom, when Python reaches it**:

In [18]:
print("before the class")
class Talkative:
    print("   ...running the class body right now!")
    x = 1 + 1
print("after the class, Talkative.x =", Talkative.x)

before the class
   ...running the class body right now!
after the class, Talkative.x = 2


## 8. Who creates classes? Meet `type`

Every object is made by a class. A class is an object. So **which class made the class?**

In [19]:
print(type(5))          # 5 was made by int
print(type(int))        # int was made by ... type
print(type(Robot))      # our own class was ALSO made by type
print(type(str), type(list), type(Dog))

<class 'int'>
<class 'type'>
<class 'type'>
<class 'type'> <class 'type'> <class 'type'>


`type` is the **class of classes**. Every class you have ever written was created by `type`. That has a name:

> A class whose instances are classes is called a **metaclass**. **`type` is Python's built-in, default metaclass.**

`type` has two jobs depending on how many arguments you give it:

| Call | What it does |
|---|---|
| `type(obj)` (1 argument) | Tells you the class of `obj` |
| `type(name, bases, namespace)` (3 arguments) | **Creates a new class** |

### Building a class without the `class` keyword

These two definitions produce the same thing:

In [20]:
# The normal way
class PersonA:
    species = "human"
    def hello(self):
        return f"Hi, I am a {self.species}"

# The "behind the scenes" way
def hello(self):
    return f"Hi, I am a {self.species}"

PersonB = type(
    "PersonB",                               # name    → becomes __name__
    (object,),                               # bases   → parent classes (a tuple)
    {"species": "human", "hello": hello},    # namespace → becomes the class __dict__
)

print(PersonA().hello(), "|", PersonB().hello())
print(type(PersonA), type(PersonB))

Hi, I am a human | Hi, I am a human
<class 'type'> <class 'type'>


So when Python executes a `class` statement it roughly does this:

```
class Child(Parent):          1. collect name "Child" and bases (Parent,)
    x = 1                     2. create an empty dict and run the body inside it
    def f(self): ...             → namespace = {"x": 1, "f": <function f>}
                              3. Child = type("Child", (Parent,), namespace)
                              4. bind the name Child to that new class object
```

The `class` keyword is just friendlier syntax for "call the metaclass".

## 9. The chicken-and-egg: `type` and `object`

Two special objects sit at the top of everything. To understand them, keep **two different relationships** apart. Mixing them up is the cause of almost all confusion here:

| Relationship | Plain English | How to check | Example |
|---|---|---|---|
| **instance of** | "was **made by**" | `type(x)`, `isinstance(x, C)` | `5` was made by `int` |
| **subclass of** | "**inherits from** / is a kind of" | `C.__bases__`, `issubclass(C, P)` | `bool` inherits from `int` |

Now let's ask both questions about everything:

In [21]:
for thing in [5, True, int, bool, Robot, object, type]:
    name = thing.__name__ if isinstance(thing, type) else repr(thing)
    bases = getattr(thing, "__bases__", "(not a class)")
    print(f"{name:7}  made by: {type(thing).__name__:5}  inherits from: {bases}")

5        made by: int    inherits from: (not a class)
True     made by: bool   inherits from: (not a class)
int      made by: type   inherits from: (<class 'object'>,)
bool     made by: type   inherits from: (<class 'int'>,)
Robot    made by: type   inherits from: (<class 'object'>,)
object   made by: type   inherits from: ()
type     made by: type   inherits from: (<class 'object'>,)


Read the last two lines carefully:

- **`object`** is **made by** `type`, and **inherits from** nothing (`()`): it is the root of the *inheritance* tree.
- **`type`** is **made by** `type` (itself!), and **inherits from** `object`.

In [22]:
print("type(type) is type     :", type(type) is type)        # type made itself
print("type(object) is type   :", type(object) is type)      # object was made by type
print("issubclass(type, object):", issubclass(type, object)) # type is a kind of object
print("issubclass(object, type):", issubclass(object, type)) # but object is NOT a kind of type
print("isinstance(type, object):", isinstance(type, object)) # type is an object
print("isinstance(object, type):", isinstance(object, type)) # object is a class

type(type) is type     : True
type(object) is type   : True
issubclass(type, object): True
issubclass(object, type): False
isinstance(type, object): True
isinstance(object, type): True


Here is the whole picture. Solid arrows go **up the inheritance tree** ("inherits from"); dashed arrows point to **who made it** ("instance of"):

```
                 INHERITANCE  (subclass of)
                 ───────────────────────────
                         object   ◀── root of all classes; inherits from nothing
                        ▲  ▲  ▲
            ┌───────────┘  │  └─────────────┐
           int            Robot            type   ◀── type inherits from object
            ▲
           bool

                 CREATION  (instance of / "made by")
                 ───────────────────────────────────
     5     ┄┄made by┄┄▶  int    ┄┄made by┄┄▶  type
     True  ┄┄made by┄┄▶  bool   ┄┄made by┄┄▶  type
     r     ┄┄made by┄┄▶  Robot  ┄┄made by┄┄▶  type
                         object ┄┄made by┄┄▶  type
                                              type ┄┄made by┄┄▶ type  (itself)
```

**How can `type` make itself, and `object` be made by `type` while `type` inherits from `object`?** In normal Python code this circle would be impossible. `type` and `object` are not created by Python code at all: they are written in C inside the interpreter, and when Python starts up it wires them together by hand. After that, everything else is built on top of them by the normal rules. So:

- If you climb the **"made by"** chain from any object, you always end at **`type`** (and stop, because `type` made itself).
- If you climb the **"inherits from"** chain from any class, you always end at **`object`**.

In [23]:
def made_by_chain(x):
    chain = [x]
    while type(chain[-1]) is not chain[-1]:   # stop when something made itself (type)
        chain.append(type(chain[-1]))
    return " → ".join(c.__name__ if isinstance(c, type) else repr(c) for c in chain)

print(made_by_chain(5))
print(made_by_chain(r))
print("inherits:", [c.__name__ for c in bool.__mro__])   # bool → int → object

5 → int → type
<__main__.Robot object at 0x10a4eecf0> → Robot → type
inherits: ['bool', 'int', 'object']


## 10. Metaclasses: the class of a class

We now have three levels, and the same rule ("you are made by your class") applies at each one:

```
  Level 3   METACLASS    type            "factory of factories"
                │ makes
  Level 2   CLASS        Robot           "factory"
                │ makes
  Level 1   INSTANCE     Robot("R2")     "product"
```

| You want to control… | You write… |
|---|---|
| how **instances** behave | methods in the **class** |
| how **classes** behave (how they are created, what attributes they get, what calling them does) | methods in the **metaclass** |

A **custom metaclass** is simply a class that **inherits from `type`**. You attach it to a class with `metaclass=`. Since `type` builds every class, inheriting from it lets you step in while a class is being built.

In [24]:
class LoudMeta(type):
    def __new__(mcls, name, bases, namespace):
        print(f"LoudMeta is building class {name!r} with attributes {[k for k in namespace if not k.startswith('__')]}")
        cls = super().__new__(mcls, name, bases, namespace)   # let type do the real work
        cls.created_by = "LoudMeta"                           # then customise the new class
        return cls

class Animal(metaclass=LoudMeta):
    sound = "..."
    def speak(self):
        return self.sound

print(type(Animal))            # Animal's class is now LoudMeta, not type
print(Animal.created_by)

LoudMeta is building class 'Animal' with attributes ['sound', 'speak']
<class '__main__.LoudMeta'>
LoudMeta


Notice the message printed **when the class was defined**, before we created any instance. Metaclass code runs at **class-creation time**.

The metaclass is **inherited**: subclasses of `Animal` are also built by `LoudMeta`:

In [25]:
class Cow(Animal):
    sound = "moo"

print(type(Cow), Cow().speak(), Cow.created_by)

LoudMeta is building class 'Cow' with attributes ['sound']
<class '__main__.LoudMeta'> moo LoudMeta


Compare the two `__new__` methods you have now seen:

| | `Class.__new__(cls, ...)` | `Metaclass.__new__(mcls, name, bases, namespace)` |
|---|---|---|
| Runs when | you create an **instance**: `Dog("Rex")` | you create a **class**: a `class` statement |
| First argument | the class (`cls`) | the metaclass (`mcls`) |
| Returns | a new instance | a new class |

### Metaclass methods belong to the class, not to its instances

Just like methods on a class are available on its instances, methods on a **metaclass** are available on the **classes** it made, but not on those classes' instances:

In [26]:
class CountingMeta(type):
    def describe(cls):                         # `cls` is a class made by this metaclass
        return f"{cls.__name__} has {len([k for k in vars(cls) if not k.startswith('__')])} public attributes"

class Shape(metaclass=CountingMeta):
    sides = 0
    def area(self): ...

print(Shape.describe())                        # works on the class
print(hasattr(Shape(), "describe"))            # not on an instance

Shape has 2 public attributes
False


### Calling a class is calling its metaclass

Back in section 5 we said `Dog("Rex")` really runs `type(Dog).__call__(Dog, "Rex")`. That `__call__` is a method on the **metaclass**, and it is what calls `__new__` and then `__init__`. Overriding it lets a metaclass control instance creation, for example to allow only one instance:

In [27]:
class OnlyOneMeta(type):
    _instances = {}
    def __call__(cls, *args, **kwargs):
        if cls not in cls._instances:
            print(f"  first call: creating the {cls.__name__} instance")
            cls._instances[cls] = super().__call__(*args, **kwargs)   # normal __new__ + __init__
        return cls._instances[cls]

class AppConfig(metaclass=OnlyOneMeta):
    def __init__(self):
        self.debug = True

c1 = AppConfig()
c2 = AppConfig()
print(c1 is c2)

  first call: creating the AppConfig instance
True


## 11. The complete class-creation sequence

Putting it all together, here is everything Python does for

```python
@decorator
class Child(Base, metaclass=Meta, flag=True):
    body...
```

1. **Pick the metaclass**: the explicit `metaclass=`, otherwise the metaclass of the bases (normally `type`).
2. **`Meta.__prepare__(name, bases)`** returns the empty dict the body will fill (usually a plain `dict`).
3. **Run the class body** inside that dict.
4. **`Meta.__new__(Meta, name, bases, namespace)`** creates the class object. While doing so, `type.__new__` calls
   - `__set_name__` on every attribute that defines it (used by descriptors, Module 10 · 02), and
   - `Base.__init_subclass__(Child, flag=True)` on the parent.
5. **`Meta.__init__(cls, name, bases, namespace)`** can finish setting the class up.
6. **Class decorators** are applied, bottom-up.
7. The name **`Child`** is bound to the result.

In [28]:
class TraceMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases, **kw):
        print(f"1. __prepare__      for {name}")
        return {}
    def __new__(mcls, name, bases, ns, **kw):
        print(f"3. TraceMeta.__new__ for {name}")
        return super().__new__(mcls, name, bases, ns, **kw)
    def __init__(cls, name, bases, ns, **kw):
        print(f"5. TraceMeta.__init__ for {name}")
        super().__init__(name, bases, ns)

class Field:
    def __set_name__(self, owner, attr):
        print(f"4a. __set_name__     {owner.__name__}.{attr}")

class Base(metaclass=TraceMeta):
    def __init_subclass__(cls, **kw):
        print(f"4b. __init_subclass__ for {cls.__name__} with {kw}")
        super().__init_subclass__()

def decorator(cls):
    print(f"6. decorator         on {cls.__name__}")
    return cls

print("---- defining Child ----")
@decorator
class Child(Base, flag=True):
    print("2. class body runs")
    field = Field()
print("7. Child is bound:", Child)

1. __prepare__      for Base
3. TraceMeta.__new__ for Base
5. TraceMeta.__init__ for Base
---- defining Child ----
1. __prepare__      for Child
2. class body runs
3. TraceMeta.__new__ for Child
4a. __set_name__     Child.field
4b. __init_subclass__ for Child with {'flag': True}
5. TraceMeta.__init__ for Child
6. decorator         on Child
7. Child is bound: <class '__main__.Child'>


## 12. Do you actually need a metaclass? Usually not

Metaclasses are powerful but heavy: a class can have only **one** metaclass, and combining two libraries that each use their own metaclass causes a "metaclass conflict" error. For most real goals there is a simpler tool:

| Goal | Simplest tool |
|---|---|
| Run code every time someone **subclasses** your class (registries, validation) | `__init_subclass__` |
| Add or change methods on **one** class after it is created | a **class decorator** |
| Reusable behaviour for **one attribute** | a **descriptor** / `property` |
| Change how **classes themselves** behave (`__call__`, operators on the class, the namespace) | a **metaclass** |

The plugin registry below would be a classic metaclass job, but `__init_subclass__` does it in a few lines. It is a hook on the **parent** class that runs whenever a **child** class is created:

In [29]:
class Plugin:
    registry = {}

    def __init_subclass__(cls, /, name, **kwargs):
        super().__init_subclass__(**kwargs)
        Plugin.registry[name] = cls

class CsvPlugin(Plugin, name="csv"):
    def run(self): return "writing CSV"

class JsonPlugin(Plugin, name="json"):
    def run(self): return "writing JSON"

print(Plugin.registry)
print(Plugin.registry["json"]().run())

{'csv': <class '__main__.CsvPlugin'>, 'json': <class '__main__.JsonPlugin'>}
writing JSON


### Metaclasses you already use

You have been using metaclasses without noticing. Check what made these classes:

In [30]:
from abc import ABC, abstractmethod
from enum import Enum

class Base2(ABC):
    @abstractmethod
    def run(self): ...

class Color(Enum):
    RED = 1
    GREEN = 2

print(type(Base2))      # ABCMeta: blocks instantiation while abstract methods remain
print(type(Color))      # EnumType: makes the class iterable and its members unique
print(list(Color))      # iterating over a CLASS works because its metaclass allows it

<class 'abc.ABCMeta'>
<class 'enum.EnumType'>
[<Color.RED: 1>, <Color.GREEN: 2>]


Pydantic's `BaseModel` (Module 15), Django models and SQLAlchemy's declarative classes use metaclasses too, to turn the attributes you write in a class body into fields, validators and database columns. By contrast, `@dataclass` is a **class decorator**, not a metaclass:

In [31]:
from dataclasses import dataclass

@dataclass
class Point:
    x: int
    y: int

print(type(Point), Point(1, 2))   # a plain class made by type, with methods added by the decorator

<class 'type'> Point(x=1, y=2)


## 13. Misconceptions to drop

| ❌ Myth | ✅ Reality |
|---|---|
| "A variable is a box holding a value." | A variable is a **name tag** pointing at an object; many names can point at one object. |
| "`is` and `==` are the same." | `==` compares **values**; `is` compares **identity** (same object). |
| "`x = x + 1` changes the number." | Ints are immutable: it creates a **new** object and moves the name. |
| "Classes are just definitions, not real things." | Classes are **runtime objects** (instances of `type`) you can pass around and modify. |
| "`__init__` creates the object." | `__new__` **creates** it; `__init__` only **initialises** it. |
| "`type` is just a function that tells you the type." | `type` is a **class**, the default **metaclass**; with 3 arguments it builds classes. |
| "`object` is the class of everything." | `object` is the **base** (top of inheritance) of every class; `type` is the class (**creator**) of every class. |
| "Metaclasses are needed for advanced OOP." | Most needs are met by `__init_subclass__`, decorators or descriptors. |

## 📝 Summary
- Every object has an **identity** (`id`, `is`), a **type** (`type()`) and a **value** (`==`). Names are labels pointing at objects.
- **Everything you can store in a variable is an object**: numbers, strings, `None`, functions, modules, classes, exceptions, `type` and `object`. Keywords, names and operators are syntax, not objects.
- Every object was **made by a class**. Literals are shortcuts for calling `int`, `str`, `list`, ….
- `Class(...)` → metaclass `__call__` → `__new__` (**create**) → `__init__` (**initialise**).
- Classes are objects made by **`type`**; a `class` statement is roughly `type(name, bases, namespace)`.
- **Made by** chain always ends at `type` (which made itself). **Inherits from** chain always ends at `object`. `type` inherits from `object`; `object` is made by `type`; both are wired up in C at startup.
- A **metaclass** is a class whose instances are classes; custom ones inherit from `type` and run at class-creation time.
- Prefer `__init_subclass__`, class decorators and descriptors; use a metaclass only when you must change how classes themselves behave.

## 🏋️ Practice
1. Write `describe(x)` that prints the id, type and value of any object, and try it on `7`, `"py"`, `len` and `int`.
2. Without running it, predict the output of `a = [1]; b = a; b += [2]; print(a, a is b)`. Then predict it for `a = (1,); b = a; b += (2,); print(a, a is b)`. Run both and explain the difference.
3. Create a class `Book` with a `title` attribute and a `read()` method **using only `type(...)`**, not the `class` keyword.
4. Write a function `who_made(x)` that returns the full "made by" chain as a list of names, e.g. `who_made(True)` → `['True', 'bool', 'type']`.
5. Write a metaclass `UpperAttrsMeta` that rejects any class whose public class attributes (non-dunder names) are not all lowercase.
6. Rewrite exercise 5 using `__init_subclass__` instead of a metaclass.

<details><summary>Show solutions</summary>

```python
# 1
def describe(x):
    print(f"id={id(x)}  type={type(x).__name__}  value={x!r}")

for obj in (7, "py", len, int):
    describe(obj)

# 2  Lists are mutable: += changes the SAME list, so a sees it → [1, 2] True
a = [1]; b = a; b += [2]; print(a, a is b)
#    Tuples are immutable: += builds a NEW tuple for b, a is untouched → (1,) False
a = (1,); b = a; b += (2,); print(a, a is b)

# 3
def read(self):
    return f"Reading {self.title}"

def book_init(self, title):
    self.title = title

Book = type("Book", (), {"__init__": book_init, "read": read})
print(Book("Dune").read(), type(Book))

# 4
def who_made(x):
    chain = [x]
    while type(chain[-1]) is not chain[-1]:
        chain.append(type(chain[-1]))
    return [c.__name__ if isinstance(c, type) else repr(c) for c in chain]

print(who_made(True))     # ['True', 'bool', 'type']

# 5
class UpperAttrsMeta(type):
    def __new__(mcls, name, bases, ns):
        bad = [k for k in ns if not k.startswith("__") and k != k.lower()]
        if bad:
            raise TypeError(f"{name}: attribute names must be lowercase, got {bad}")
        return super().__new__(mcls, name, bases, ns)

class Good(metaclass=UpperAttrsMeta):
    speed = 1
try:
    class Bad(metaclass=UpperAttrsMeta):
        Speed = 1
except TypeError as e:
    print(e)

# 6  (checks subclasses of LowerBase, not LowerBase itself)
class LowerBase:
    def __init_subclass__(cls, **kwargs):
        super().__init_subclass__(**kwargs)
        bad = [k for k in vars(cls) if not k.startswith("__") and k != k.lower()]
        if bad:
            raise TypeError(f"{cls.__name__}: attribute names must be lowercase, got {bad}")

class Fine(LowerBase):
    speed = 1
try:
    class Wrong(LowerBase):
        Speed = 1
except TypeError as e:
    print(e)
```
</details>